In [1]:
# load libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, roc_auc_score,
                              RocCurveDisplay, ConfusionMatrixDisplay, confusion_matrix)
from lightgbm import LGBMRegressor
import lightgbm as lgb
from sklearn.model_selection import RandomizedSearchCV

In [2]:
# load in data
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [3]:
# save separate predictor and response variables
X_train = train[['CTDTEMP_ITS90', 'Salinity_PSS78', 'Depth', 'TA']]
y_train = train['DIC']
X_test = test[['CTDTEMP_ITS90', 'Salinity_PSS78', 'Depth', 'TA']]

# scale the data
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Light Gradient Boosting Machine (LightGBM)

Similar to XGBoost, the LightGBM model uses tree-based algorithms to create the best predictive model. The LightGBM model uses leaf-wise splitting instead of level-wise splitting. This makes asymmetric decision trees by splitting the nodes with the highest impurity or uncertainty rates.

Hyperparameters of a LightGBM:

    - n_estimators: Number of boosted trees to fit. 
    - learning_rate: How much each tree influences final predictions
    - num_leaves: How complex a tree can be/ How many leaves a tree has. 

More hyperparameters are available for tuning but these are the ones we found to be most influential on the R2. 

Using `GridSearchCV()` which iterates through all of the possible combinations of the parameters was taking too much computing power and far too long. Instead the `RandomizedSearchCV()` was utilized to test a fraction of the possible combinations of the parameters, in this case we tried 30 combinations. 

**This gave us our best score!**

In [6]:
# make a parameter grid for the lgb model
param_grid = {
    'n_estimators':   [100, 300, 500, 700, 800],
    'learning_rate':  [0.01, 0.05, 0.1, 0.2],
    'num_leaves':     [10, 20, 31, 50] # for some reason 31 is used in most of the documentation
}

# define the lgb model to find the best parameters
grid_lgb = RandomizedSearchCV(
    lgb.LGBMRegressor(verbose= -1, # controls logging information which -1 stops altogether
                      n_jobs = 1),
    param_grid,
    n_iter=30,       # 30 random combos 
    cv=5,
    scoring='r2',
    n_jobs=-1,
    random_state=42
)

# fit the lgb model to the data 
grid_lgb.fit(X_train_scaled, y_train)

# predict the test 
y_test_pred_lgb = grid_lgb.predict(X_test_scaled)

# access the lgb model's accuracy
lgb_best = grid_lgb.best_estimator_

print(f"Best parameters: {grid_lgb.best_params_}")
print(f"CV R2: {grid_lgb.best_score_:.4f}")  

Best parameters: {'num_leaves': 31, 'n_estimators': 100, 'learning_rate': 0.1}
CV R2: 0.9751


The `RandomSearchCV()` found the R2 is best for the LightGBM model when the number of leaves is 31, the number of boosted trees is 100, and the learning rate is 0.1.

In [7]:
submission = pd.DataFrame({"id": test["id"], "DIC": y_test_pred_lgb})
submission.to_csv("submission.csv", index=False)

In [ ]:
submission

# References
